# Recommendation engine: full run order for Google Colab

Colab wipes `/content` when the runtime disconnects, so the code and every result live in Google Drive. Run the cells top to bottom. After each step the outputs are copied to Drive, so a disconnect only costs you the current step.

Use a Drive folder name **without spaces**. Steps 3 and 5 are much faster on a GPU runtime but work on CPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/recsys'     # code + saved results (no spaces in the name)
WORK = '/content/work'                        # fast local disk for big files
import os
os.makedirs(PROJECT, exist_ok=True); os.makedirs(WORK, exist_ok=True)
print(PROJECT)

## 0. Code and libraries
Copy the repo into `PROJECT` first (unzip the project zip there, or `git clone` your GitHub repo).

In [ ]:
%cd {PROJECT}
!ls
!pip install -q pyspark faiss-cpu tensorflow-recommenders tf-keras pyarrow redis gradio

In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'      # TensorFlow Recommenders needs Keras 2

## 1. Download MovieLens 25M, clean and split by time

In [ ]:
!wget -q -P {WORK} https://files.grouplens.org/datasets/movielens/ml-25m.zip
!unzip -q -o {WORK}/ml-25m.zip -d {WORK}/raw
!python {PROJECT}/src/data/prepare_data.py --raw {WORK}/raw/ml-25m --out {WORK}/processed
!cp -r {WORK}/processed {PROJECT}/processed

## 2. Spark features

In [ ]:
!python {PROJECT}/src/features/build_features.py --data {WORK}/processed --out {WORK}/features
!cp -r {WORK}/features {PROJECT}/features

## 3. Retrieval: two-tower model (about 27 min per epoch on CPU)

In [ ]:
!python {PROJECT}/src/retrieval/train_two_tower.py --data {WORK}/processed --out {WORK}/retrieval --epochs 3 --batch 4096
!cp -r {WORK}/retrieval {PROJECT}/retrieval
!cat {PROJECT}/retrieval/metrics.json

## 4. FAISS indexes and speed benchmark

In [ ]:
!python {PROJECT}/src/retrieval/build_faiss_index.py --emb {WORK}/retrieval --out {WORK}/faiss
!cp -r {WORK}/faiss {PROJECT}/faiss

## 5. Ranker (DCN trained on retrieval candidates)

In [ ]:
!python {PROJECT}/src/ranking/train_candidate_ranker.py --data {WORK}/processed --features {WORK}/features \
    --retrieval {WORK}/retrieval --faiss {WORK}/faiss --out {WORK}/ranking_v3
!cp -r {WORK}/ranking_v3 {PROJECT}/ranking_v3

## 6. Recommender demo with a Redis cache

In [ ]:
!apt-get -qq install -y redis-server > /dev/null
!redis-server --daemonize yes > /dev/null

In [ ]:
ARGS = f"--artifacts-root {WORK} --data processed --features features --retrieval retrieval --faiss faiss --ranking ranking_v3"
!python {PROJECT}/src/serving/recommender.py {ARGS} --redis localhost:6379 --demo

## 7. Replay A/B test and latency benchmark (results go straight to Drive)

In [ ]:
!python {PROJECT}/src/serving/ab_test.py {ARGS} --out {PROJECT}/outputs/ab_test
!python {PROJECT}/src/serving/benchmark.py {ARGS} --redis localhost:6379 --out {PROJECT}/outputs/benchmark

## 8. Export the ranker for Triton
Colab has no Docker, so the Triton server itself cannot run here. This step builds the model repository and checks ONNX against the Keras model (difference should be below 1e-4).

In [ ]:
!pip install -q onnx onnxruntime tf2onnx
!python {PROJECT}/src/serving/export_triton.py --artifacts-root {WORK} --data processed --retrieval retrieval --ranking ranking_v3 --out {PROJECT}/model_repository

## 9. Demo app (public link for a limited time)

In [ ]:
!python {PROJECT}/app/demo_app.py {ARGS} --redis localhost:6379 --share